# 02 · Modelling, uplift and the business case (Google Colab version)

**Group 11 · Track 3 Project Showcase · Topic A: Customer Churn Prediction with Simple Uplift Extension**

This notebook runs on its own in Google Colab: **Runtime → Run all**. The setup cells below install the libraries,
recreate the project's `ml/` code, download the public IBM Telco dataset and run the pipeline
(about 4–6 minutes on a standard Colab runtime). The notebook itself starts at the next big heading.

The same notebook lives in the project as `notebooks/02_modelling.ipynb`; this copy only adds setup, so every number matches the project.

## Setup 1 · Install the exact library versions the project uses

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'scikit-learn==1.6.1', 'xgboost==3.4.1', 'shap==0.54.1', 'scikit-uplift==0.5.1'], check=True)
    print("Installed:", 'scikit-learn==1.6.1', 'xgboost==3.4.1', 'shap==0.54.1', 'scikit-uplift==0.5.1')
else:
    print("Not on Colab: using the libraries already installed.")

## Setup 2 · Recreate the project code

These files are copied unchanged from the project's `ml/` folder (and `params.yaml`, the business assumptions).

In [ ]:
from pathlib import Path
for d in ['ml/models', 'ml/results/figures', 'data/raw', 'data/processed', 'backend/data', 'notebooks/figures']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Project folders ready in', Path.cwd())

In [ ]:
%%writefile ml/download_data.py
"""
Download the IBM Telco Customer Churn dataset into data/raw/.

Why a script instead of committing the CSV?
  - Raw data stays out of git (project rule).
  - Anyone can rebuild the data folder with one command.

Run from the project root:
    python ml/download_data.py
"""
from pathlib import Path
import urllib.request

# Public copy of the dataset published by IBM on GitHub.
URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

# Build paths relative to this file so the script works from any folder.
PROJECT_ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "telco_churn.csv"


def main() -> None:
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    if RAW_PATH.exists():
        print(f"Already downloaded: {RAW_PATH}")
        return
    print(f"Downloading from {URL} ...")
    urllib.request.urlretrieve(URL, RAW_PATH)
    print(f"Saved to {RAW_PATH} ({RAW_PATH.stat().st_size / 1024:.0f} KB)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/preprocess.py
"""
Data cleaning + preprocessing for the Telco churn dataset.

Two separate steps:
  1. clean_data()          -> fixes the raw data (types, missing values, target).
                              Output is still human-readable (e.g. "Month-to-month").
  2. build_preprocessor()  -> scikit-learn transformer that ENCODES categories
                              (one-hot) and SCALES numbers (standardisation).

Why keep step 2 separate?
  The scaler must learn its mean/std from the TRAINING data only. If we scaled
  the whole dataset before splitting, information from the test set would
  "leak" into training. Putting the scaler inside a scikit-learn Pipeline
  (Phase 3) avoids this and lets the backend reuse the exact same steps.

Run from the project root to create data/processed/telco_clean.csv:
    python ml/preprocess.py
"""
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "telco_churn.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "telco_clean.csv"

ID_COL = "customerID"
TARGET = "Churn"

# Numeric columns -> get scaled.
NUMERIC_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]

# Already 0/1 -> used as-is (no encoding or scaling needed).
BINARY_NUMERIC_COLS = ["SeniorCitizen"]

# Text categories -> get one-hot encoded.
CATEGORICAL_COLS = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]

FEATURE_COLS = NUMERIC_COLS + BINARY_NUMERIC_COLS + CATEGORICAL_COLS


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy of the raw Telco data."""
    df = df.copy()

    # 1) TotalCharges is stored as text and has 11 blank values.
    #    All 11 blanks are brand-new customers (tenure = 0) who have not been
    #    billed yet, so the correct value is 0 (not "missing").
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)

    # 2) "No internet service" / "No phone service" mean the same as "No" for
    #    the add-on columns. The InternetService / PhoneService columns already
    #    record whether the customer has the base service, so merging these
    #    removes duplicate information and gives fewer one-hot columns.
    df = df.replace({"No internet service": "No", "No phone service": "No"})

    # 3) Target: "Yes"/"No" -> 1/0 so models can use it.
    if TARGET in df.columns:
        df[TARGET] = (df[TARGET] == "Yes").astype(int)

    return df


def build_preprocessor() -> ColumnTransformer:
    """
    Encoding + scaling step, to be placed at the start of a model Pipeline.
      - numbers      -> StandardScaler (mean 0, std 1)
      - categories   -> OneHotEncoder (one 0/1 column per category)
      - 0/1 columns  -> passed through unchanged
    handle_unknown="ignore" means an unseen category at prediction time
    (e.g. a typo from the web form) becomes all zeros instead of crashing.
    """
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_COLS),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CATEGORICAL_COLS),
            ("bin", "passthrough", BINARY_NUMERIC_COLS),
        ]
    )


def load_clean() -> pd.DataFrame:
    """Load the cleaned CSV (create it first with `python ml/preprocess.py`)."""
    return pd.read_csv(CLEAN_PATH)


RANDOM_STATE = 42   # fixed seed -> same split / same results every run
TEST_SIZE = 0.20    # 80% train, 20% test


def split_data(df: pd.DataFrame):
    """
    80/20 train/test split, STRATIFIED on Churn so both parts keep the
    same churn rate (about 26.5%). Every phase uses this function, so all
    models are evaluated on exactly the same test customers.
    """
    return train_test_split(
        df, test_size=TEST_SIZE, stratify=df[TARGET], random_state=RANDOM_STATE
    )


def main() -> None:
    raw = pd.read_csv(RAW_PATH)
    clean = clean_data(raw)
    CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
    clean.to_csv(CLEAN_PATH, index=False)
    print(f"Raw shape:   {raw.shape}")
    print(f"Clean shape: {clean.shape}")
    print(f"Missing values after cleaning: {int(clean.isna().sum().sum())}")
    print(f"Churn rate: {clean[TARGET].mean():.1%}")
    print(f"Saved to {CLEAN_PATH}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/train_classical.py
"""
Phase 3: train and compare three classical churn models.

  1. Logistic Regression  (simple, linear, easy to explain)
  2. Random Forest        (many decision trees voting together)
  3. XGBoost              (trees built one after another, each fixing the
                           previous trees' mistakes: "gradient boosting")

Each model is a scikit-learn Pipeline:  [encode + scale]  ->  [model]
so the preprocessing is learned from the TRAINING data only (no leakage)
and the saved file contains everything needed to predict on raw input.

Class imbalance (26.5% churners) is handled with class weights: mistakes
on churners count about 3x more during training, so the models don't just
predict "no churn" for everyone.

Calibration: class weights make the raw probabilities too HIGH (the model
predicted 39% churn on average when the real rate is 26.5%). Ranking is still
good, but the app shows these numbers as "chance of staying", so the chosen
model is wrapped in CalibratedClassifierCV (sigmoid / Platt scaling, 5-fold).
That keeps the ranking (ROC-AUC unchanged) and makes the probabilities honest.

Outputs:
  ml/models/best_churn_model.joblib     best pipeline, CALIBRATED (used for all predictions)
  ml/models/churn_model_uncalibrated.joblib  same model before calibration (used for SHAP)
  ml/results/figures/calibration_curve.png   predicted vs actual churn, before/after
  ml/results/classical_metrics.json     metrics for all models (used by the API)
  ml/results/shap_importance.json       top churn drivers (used by the dashboard)
  ml/results/figures/*.png              ROC curves, confusion matrix, SHAP plots

Run from the project root:
    python ml/train_classical.py
"""
import json
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")  # save charts to files, no pop-up windows
import matplotlib.pyplot as plt
import numpy as np
import shap
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, brier_score_loss, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from preprocess import (FEATURE_COLS, PROJECT_ROOT, RANDOM_STATE, TARGET,
                        build_preprocessor, load_clean, split_data)

MODEL_DIR = PROJECT_ROOT / "ml" / "models"
RESULTS_DIR = PROJECT_ROOT / "ml" / "results"
FIG_DIR = RESULTS_DIR / "figures"
THRESHOLD = 0.5  # probability above this -> predict "churn"
CALIBRATION = "sigmoid"  # Platt scaling: keeps the ranking, fixes the probability scale


def make_models(pos_weight: float) -> dict:
    """
    The three models with simple, reasonable settings (no heavy tuning,
    to keep the project explainable). pos_weight = (#stayers / #churners).
    """
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,        # 300 trees
            max_depth=8,             # limit depth -> less overfitting, smaller file
            min_samples_leaf=5,
            class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=4,             # shallow trees
            learning_rate=0.05,      # small steps -> more stable
            subsample=0.8,           # each tree sees 80% of rows
            colsample_bytree=0.8,    # ...and 80% of columns
            scale_pos_weight=pos_weight,  # XGBoost's version of class weights
            eval_metric="logloss",
            n_jobs=-1, random_state=RANDOM_STATE),
    }


def evaluate(y_true, prob) -> dict:
    """Imbalance-friendly metrics. Accuracy is deliberately not the headline."""
    pred = (prob >= THRESHOLD).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, prob),       # ranking quality, threshold-free
        "precision": precision_score(y_true, pred),   # of flagged customers, % that churn
        "recall": recall_score(y_true, pred),         # of real churners, % we caught
        "f1": f1_score(y_true, pred),                 # balance of precision & recall
    }


def pretty_name(feature: str) -> str:
    """'cat__Contract_Month-to-month' -> 'Contract = Month-to-month'."""
    name = feature.split("__", 1)[1]
    for col in FEATURE_COLS:
        if name.startswith(col + "_"):
            return f"{col} = {name[len(col) + 1:]}"
    return name


def shap_analysis(pipeline, X_test, model_name: str) -> list:
    """
    SHAP explains each prediction as a sum of feature contributions.
    Mean |SHAP value| per feature = how much that feature moves predictions
    on average, i.e. global feature importance.
    """
    pre = pipeline.named_steps["preprocess"]
    model = pipeline.named_steps["model"]
    X_t = pre.transform(X_test)
    names = [pretty_name(f) for f in pre.get_feature_names_out()]

    # Tree models -> fast exact TreeExplainer; linear model -> LinearExplainer.
    if isinstance(model, LogisticRegression):
        explainer = shap.LinearExplainer(model, X_t)
    else:
        explainer = shap.TreeExplainer(model)
    sv = explainer(X_t)
    sv.feature_names = names
    if sv.values.ndim == 3:  # Random Forest gives one set per class -> keep "churn"
        sv = sv[:, :, 1]

    # Beeswarm: every dot is a customer; right = pushes towards churn.
    plt.figure()
    shap.plots.beeswarm(sv, max_display=15, show=False)
    plt.title(f"SHAP summary ({model_name})")
    plt.tight_layout(); plt.savefig(FIG_DIR / "shap_beeswarm.png", dpi=120); plt.close("all")

    plt.figure()
    shap.plots.bar(sv, max_display=15, show=False)
    plt.title(f"SHAP feature importance ({model_name})")
    plt.tight_layout(); plt.savefig(FIG_DIR / "shap_bar.png", dpi=120); plt.close("all")

    # Direction: does a HIGH value of this feature raise or lower churn risk?
    mean_abs = np.abs(sv.values).mean(axis=0)
    importance = []
    for i in np.argsort(mean_abs)[::-1][:15]:
        x, v = sv.data[:, i], sv.values[:, i]
        corr = np.corrcoef(x, v)[0, 1] if np.std(x) > 0 else 0.0
        importance.append({
            "feature": names[i],
            "mean_abs_shap": round(float(mean_abs[i]), 4),
            "direction": "increases churn" if corr > 0 else "decreases churn",
        })
    return importance


def main() -> None:
    for d in (MODEL_DIR, FIG_DIR):
        d.mkdir(parents=True, exist_ok=True)

    # ---- 1. Data ----
    df = load_clean()
    train, test = split_data(df)
    X_train, y_train = train[FEATURE_COLS], train[TARGET]
    X_test, y_test = test[FEATURE_COLS], test[TARGET]
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    print(f"Train: {len(train)} rows | Test: {len(test)} rows | "
          f"churn rate train={y_train.mean():.1%} test={y_test.mean():.1%}\n")

    # ---- 2. Train + evaluate each model ----
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    results, pipelines, probs = {}, {}, {}
    for name, model in make_models(pos_weight).items():
        pipe = Pipeline([("preprocess", build_preprocessor()), ("model", model)])

        # 5-fold cross-validation on TRAIN only: stability check
        cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")

        pipe.fit(X_train, y_train)
        prob = pipe.predict_proba(X_test)[:, 1]
        metrics = evaluate(y_test, prob)
        metrics["cv_roc_auc_mean"] = cv_auc.mean()
        metrics["cv_roc_auc_std"] = cv_auc.std()

        results[name] = {k: round(float(v), 4) for k, v in metrics.items()}
        pipelines[name], probs[name] = pipe, prob
        print(f"{name:20s} " + "  ".join(f"{k}={v:.3f}" for k, v in results[name].items()))

    # ---- 3. Pick the best model by test ROC-AUC ----
    best_name = max(results, key=lambda n: results[n]["roc_auc"])
    best_pipe = pipelines[best_name]
    print(f"\nBest model (highest test ROC-AUC): {best_name}")

    # ---- 3b. Calibrate the chosen model's probabilities ----
    # 5-fold on TRAIN: each fold fits the model on 4/5 and learns the probability
    # correction on the held-out 1/5. The test set stays untouched.
    calibrated = CalibratedClassifierCV(clone(best_pipe), method=CALIBRATION, cv=5)
    calibrated.fit(X_train, y_train)
    raw_prob, cal_prob = probs[best_name], calibrated.predict_proba(X_test)[:, 1]
    summary = lambda p: {"mean_predicted": round(float(p.mean()), 4),
                         "brier": round(float(brier_score_loss(y_test, p)), 4),
                         "roc_auc": round(float(roc_auc_score(y_test, p)), 4)}
    calibration = {"method": CALIBRATION, "actual_churn_rate": round(float(y_test.mean()), 4),
                   "before": summary(raw_prob), "after": summary(cal_prob), "curve": {}}

    plt.figure(figsize=(6, 5))
    plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
    for label, p in [("Before (class-weighted)", raw_prob), ("After calibration", cal_prob)]:
        frac, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
        plt.plot(mean_pred, frac, "o-", label=label)
        calibration["curve"][label] = [{"predicted": round(float(a), 4), "actual": round(float(b), 4)}
                                       for a, b in zip(mean_pred, frac)]
    plt.xlabel("Predicted churn probability"); plt.ylabel("Actual churn rate")
    plt.title(f"Calibration: {best_name} (test set)"); plt.legend(loc="upper left")
    plt.tight_layout(); plt.savefig(FIG_DIR / "calibration_curve.png", dpi=120); plt.close()
    b, a = calibration["before"], calibration["after"]
    print(f"Calibration ({CALIBRATION}): mean predicted churn {b['mean_predicted']:.1%} -> "
          f"{a['mean_predicted']:.1%} (actual {calibration['actual_churn_rate']:.1%}), "
          f"Brier {b['brier']:.4f} -> {a['brier']:.4f}, ROC-AUC {b['roc_auc']:.4f} -> {a['roc_auc']:.4f}")

    joblib.dump(calibrated, MODEL_DIR / "best_churn_model.joblib")         # used for predictions
    joblib.dump(best_pipe, MODEL_DIR / "churn_model_uncalibrated.joblib")  # used for SHAP explanations

    # ---- 4. Charts: ROC curves + confusion matrix ----
    roc_points = {}
    plt.figure(figsize=(6, 5))
    for name, prob in probs.items():
        fpr, tpr, _ = roc_curve(y_test, prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC={results[name]['roc_auc']:.3f})")
        # Keep ~50 points per curve for the frontend chart
        idx = np.linspace(0, len(fpr) - 1, 50).astype(int)
        roc_points[name] = [{"fpr": round(float(fpr[i]), 4), "tpr": round(float(tpr[i]), 4)} for i in idx]
    plt.plot([0, 1], [0, 1], "k--", label="Random guess (AUC=0.5)")
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate (recall)")
    plt.title("ROC curves (test set)"); plt.legend(loc="lower right")
    plt.tight_layout(); plt.savefig(FIG_DIR / "roc_curves.png", dpi=120); plt.close()

    cm = confusion_matrix(y_test, (probs[best_name] >= THRESHOLD).astype(int))
    ConfusionMatrixDisplay(cm, display_labels=["Stayed", "Churned"]).plot(cmap="Blues")
    plt.title(f"Confusion matrix: {best_name}")
    plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=120); plt.close()

    # ---- 5. SHAP on the best model ----
    importance = shap_analysis(best_pipe, X_test, best_name)
    print("\nTop 10 churn drivers (SHAP):")
    for r in importance[:10]:
        print(f"  {r['feature']:40s} {r['mean_abs_shap']:.3f}  ({r['direction']})")

    # ---- 6. Save results as JSON for the backend / frontend ----
    with open(RESULTS_DIR / "classical_metrics.json", "w") as f:
        json.dump({
            "best_model": best_name,
            "threshold": THRESHOLD,
            "test_size": len(test),
            "test_churn_rate": round(float(y_test.mean()), 4),
            "models": results,
            "confusion_matrix_best": cm.tolist(),
            "roc_curves": roc_points,
            "calibration": calibration,
        }, f, indent=2)
    with open(RESULTS_DIR / "shap_importance.json", "w") as f:
        json.dump({"model": best_name, "top_features": importance}, f, indent=2)
    print(f"\nSaved model + results to {MODEL_DIR} and {RESULTS_DIR}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/simulate_treatment.py
"""
!!!  SIMULATED DATA  !!!
The IBM Telco dataset has NO real "received retention offer" column.
This file INVENTS one so we can demonstrate uplift modelling. Every effect
size below is an ASSUMPTION chosen by us. These are not real campaign results.

How the simulation works
------------------------
1. Random assignment (like a fair A/B test):
   each customer gets the offer (treatment = 1) with probability 50%.

2. Customers who did NOT get the offer (control) keep their REAL churn outcome.

3. Customers who DID get the offer: their outcome may change, depending on
   which (assumed) segment they belong to:
     - a real churner may be "saved" (churn 1 -> 0) with probability SAVE_RATE
     - for "sleeping dogs", a real stayer may churn BECAUSE they were
       contacted (0 -> 1) with probability BACKFIRE_RATE

Segments (checked in this order; first match wins):
    sleeping_dog  : Two-year contract AND tenure > 48 months
    lost_cause    : Fiber optic AND Electronic check AND MonthlyCharges > 90
    persuadable   : Month-to-month AND tenure <= 12 months
    m2m_other     : any other Month-to-month customer
    contract      : any other One-/Two-year contract customer

Run from the project root to print a summary of the simulated design:
    python ml/simulate_treatment.py
"""
import numpy as np
import pandas as pd

SIM_SEED = 2024          # fixed seed -> same simulated data every run
TREATMENT_SHARE = 0.5    # 50% of customers receive the offer

# ---- ASSUMED effect sizes (edit here to change the design) ----
SAVE_RATE = {            # share of real churners the offer keeps
    "sleeping_dog": 0.00,
    "lost_cause":   0.05,
    "persuadable":  0.40,
    "m2m_other":    0.20,
    "contract":     0.05,
}
BACKFIRE_RATE = {        # share of real stayers the offer pushes to churn
    "sleeping_dog": 0.03,
}


def assign_segment(df: pd.DataFrame) -> pd.Series:
    """Label every customer with one (assumed) response segment."""
    conditions = [
        (df["Contract"] == "Two year") & (df["tenure"] > 48),
        (df["InternetService"] == "Fiber optic")
        & (df["PaymentMethod"] == "Electronic check")
        & (df["MonthlyCharges"] > 90),
        (df["Contract"] == "Month-to-month") & (df["tenure"] <= 12),
        df["Contract"] == "Month-to-month",
    ]
    choices = ["sleeping_dog", "lost_cause", "persuadable", "m2m_other"]
    return pd.Series(np.select(conditions, choices, default="contract"), index=df.index)


def simulate_treatment(df: pd.DataFrame, seed: int = SIM_SEED) -> pd.DataFrame:
    """
    Return a copy of df with SIMULATED columns added:
      segment        assumed response segment
      treatment      1 = received retention offer (random, 50%)
      churn_sim      churn outcome after applying the offer's assumed effect
      save_rate / backfire_rate   the assumed effect for that customer
    The original 'Churn' column is kept unchanged (= outcome without offer).
    """
    rng = np.random.default_rng(seed)
    out = df.copy()
    out["segment"] = assign_segment(out)
    out["save_rate"] = out["segment"].map(SAVE_RATE).fillna(0.0)
    out["backfire_rate"] = out["segment"].map(BACKFIRE_RATE).fillna(0.0)

    out["treatment"] = (rng.random(len(out)) < TREATMENT_SHARE).astype(int)

    u = rng.random(len(out))  # one random draw per customer
    saved = (out["treatment"] == 1) & (out["Churn"] == 1) & (u < out["save_rate"])
    backfired = (out["treatment"] == 1) & (out["Churn"] == 0) & (u < out["backfire_rate"])

    out["churn_sim"] = out["Churn"]
    out.loc[saved, "churn_sim"] = 0
    out.loc[backfired, "churn_sim"] = 1

    # GROUND TRUTH (only possible because this is a simulation):
    # expected number of churners saved if we contact this customer.
    #   real churner -> saved with probability save_rate      (+)
    #   real stayer  -> pushed to churn with backfire_rate    (-)
    # Used ONLY to evaluate rankings, never as a model input.
    out["true_effect"] = out["Churn"] * out["save_rate"] - (1 - out["Churn"]) * out["backfire_rate"]
    return out


def true_uplift(base_churn_prob: pd.Series, save_rate: pd.Series, backfire_rate: pd.Series) -> pd.Series:
    """
    Expected reduction in churn probability if we contact the customer
    (positive = offer helps). Known only because WE designed the simulation:
        uplift = P(churn) * save_rate  -  P(stay) * backfire_rate
    """
    return base_churn_prob * save_rate - (1 - base_churn_prob) * backfire_rate


if __name__ == "__main__":
    from preprocess import load_clean

    sim = simulate_treatment(load_clean())
    print("=== SIMULATED treatment design (assumptions, not real data) ===\n")
    print(f"Treated: {sim.treatment.sum()} ({sim.treatment.mean():.1%})   "
          f"Control: {(1 - sim.treatment).sum()}\n")

    g = sim.groupby("segment")
    table = pd.DataFrame({
        "customers": g.size(),
        "share": g.size() / len(sim),
        "real_churn_rate": g["Churn"].mean(),
        "assumed_save_rate": g["save_rate"].first(),
        "assumed_backfire": g["backfire_rate"].first(),
        "churn_control": sim[sim.treatment == 0].groupby("segment")["churn_sim"].mean(),
        "churn_treated": sim[sim.treatment == 1].groupby("segment")["churn_sim"].mean(),
    })
    table["observed_uplift"] = table["churn_control"] - table["churn_treated"]
    order = ["persuadable", "m2m_other", "lost_cause", "contract", "sleeping_dog"]
    print(table.loc[order].round(3).to_string(), "\n")

    ctrl = sim.loc[sim.treatment == 0, "churn_sim"].mean()
    trt = sim.loc[sim.treatment == 1, "churn_sim"].mean()
    print(f"Overall churn  control={ctrl:.3f}  treated={trt:.3f}  "
          f"-> average offer effect = {ctrl - trt:.3f} ({(ctrl - trt) * 100:.1f} percentage points)")


In [ ]:
%%writefile ml/train_uplift.py
"""
Phase 4: uplift modelling with a T-learner.

!!! Uses the SIMULATED retention-offer data from simulate_treatment.py !!!

Churn model vs uplift model: what's the difference?
  - Churn model  asks: "WHO is likely to leave?"
  - Uplift model asks: "WHO will stay BECAUSE we contacted them?"
  A customer who will leave no matter what (a lost cause) has high churn risk
  but zero uplift, so contacting them wastes money.

T-learner ("Two models"):
  1. Model T: trained only on customers who GOT the offer      -> P(stay | offer)
  2. Model C: trained only on customers who did NOT get it     -> P(stay | no offer)
  3. Uplift score = P(stay | offer) - P(stay | no offer)
     positive = offer helps, ~0 = no effect, negative = offer backfires.

Evaluation (on the same 20% test customers as Phase 3):
  - Qini / uplift curves (standard uplift metrics, from scikit-uplift)
  - Top-20% question: if we can only contact 20% of customers, does ranking
    by churn RISK or by UPLIFT save more customers?

Outputs:
  ml/models/uplift_model.joblib        {"treated": pipeline, "control": pipeline}
  ml/results/uplift_metrics.json       results for the API / frontend
  ml/results/figures/qini_curves.png, uplift_gain.png, uplift_by_decile.png

Run from the project root (after train_classical.py):
    python ml/train_uplift.py
"""
import json

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklift.metrics import qini_auc_score, qini_curve, uplift_at_k
from xgboost import XGBClassifier

from preprocess import FEATURE_COLS, PROJECT_ROOT, RANDOM_STATE, build_preprocessor, load_clean, split_data
from simulate_treatment import simulate_treatment

MODEL_DIR = PROJECT_ROOT / "ml" / "models"
RESULTS_DIR = PROJECT_ROOT / "ml" / "results"
FIG_DIR = RESULTS_DIR / "figures"
TOP_K = 0.20  # we can contact only the top 20% of customers


class TLearner:
    """Minimal T-learner: one model per group, uplift = difference."""

    def __init__(self, base_model):
        self.model_treated = Pipeline([("preprocess", build_preprocessor()), ("model", clone(base_model))])
        self.model_control = Pipeline([("preprocess", build_preprocessor()), ("model", clone(base_model))])

    def fit(self, X, y, treatment):
        self.model_treated.fit(X[treatment == 1], y[treatment == 1])
        self.model_control.fit(X[treatment == 0], y[treatment == 0])
        return self

    def predict_uplift(self, X):
        p_stay_offer = self.model_treated.predict_proba(X)[:, 1]
        p_stay_no_offer = self.model_control.predict_proba(X)[:, 1]
        return p_stay_offer - p_stay_no_offer


def base_models() -> dict:
    """Two candidate base learners for the T-learner."""
    return {
        "T-learner (Logistic Regression)": LogisticRegression(max_iter=1000),
        "T-learner (XGBoost)": XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
            eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE),
    }


def cv_qini(base_model, X, y, t, n_splits=5) -> float:
    """5-fold CV Qini on TRAIN data only, so we pick a base learner without touching the test set."""
    strata = t * 2 + y  # keep treatment/outcome mix the same in every fold
    scores = []
    for tr, va in StratifiedKFold(n_splits, shuffle=True, random_state=RANDOM_STATE).split(X, strata):
        m = TLearner(base_model).fit(X.iloc[tr], y.iloc[tr], t.iloc[tr])
        scores.append(qini_auc_score(y.iloc[va], m.predict_uplift(X.iloc[va]), t.iloc[va]))
    return float(np.mean(scores))


def true_gain_curve(score, true_effect, n_points=21):
    """Cumulative TRUE expected customers saved when contacting the top x% by `score`."""
    order = np.argsort(-score)
    cum = np.concatenate([[0], np.cumsum(true_effect[order])])
    fracs = np.linspace(0, 1, n_points)
    return fracs, cum[(fracs * len(score)).round().astype(int)]


def main() -> None:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

    # ---- 1. Simulated data, same train/test customers as Phase 3 ----
    sim = simulate_treatment(load_clean())
    train, test = split_data(sim)
    # Outcome for uplift: 1 = customer STAYED (so positive uplift = good)
    y_train, y_test = 1 - train["churn_sim"], 1 - test["churn_sim"]
    t_train, t_test = train["treatment"], test["treatment"]
    X_train, X_test = train[FEATURE_COLS], test[FEATURE_COLS]

    # ---- 2. Choose the base learner by cross-validated Qini (train only) ----
    print("5-fold CV Qini on train (higher = better):")
    cv_scores = {}
    for name, base in base_models().items():
        cv_scores[name] = cv_qini(base, X_train, y_train, t_train)
        print(f"  {name:35s} {cv_scores[name]:.4f}")
    chosen = max(cv_scores, key=cv_scores.get)
    print(f"Chosen: {chosen}\n")

    # ---- 3. Fit all T-learners on full train, score the test set ----
    scores = {}
    for name, base in base_models().items():
        m = TLearner(base).fit(X_train, y_train, t_train)
        scores[name] = m.predict_uplift(X_test)
        if name == chosen:
            joblib.dump({"treated": m.model_treated, "control": m.model_control, "name": name},
                        MODEL_DIR / "uplift_model.joblib")

    # Churn-risk ranking = the Phase 3 best classifier (trained on real churn)
    churn_model = joblib.load(MODEL_DIR / "best_churn_model.joblib")
    scores["Churn risk (Random Forest)"] = churn_model.predict_proba(X_test)[:, 1]
    rng = np.random.default_rng(RANDOM_STATE)
    scores["Random"] = rng.random(len(test))

    # ---- 4. Evaluate every ranking ----
    true_effect = test["true_effect"].to_numpy()
    k = int(round(TOP_K * len(test)))
    best_possible = np.sort(true_effect)[::-1][:k].sum()
    results = {}
    for name, s in scores.items():
        top = np.argsort(-s)[:k]
        results[name] = {
            "qini_auc": round(float(qini_auc_score(y_test, s, t_test)), 4),
            # observed: stay-rate(treated) - stay-rate(control) inside the top 20%
            "observed_uplift_at_20": round(float(uplift_at_k(y_test, s, t_test, strategy="overall", k=TOP_K)), 4),
            # ground truth (simulation only): expected customers saved by contacting the top 20%
            "true_saved_top20": round(float(true_effect[top].sum()), 1),
            "segments_top20": test.iloc[top]["segment"].value_counts().to_dict(),
        }
    print(f"Test set: {len(test)} customers | top 20% = {k} | best possible saved = {best_possible:.1f}\n")
    print(pd.DataFrame(results).T.drop(columns="segments_top20").to_string(), "\n")
    for name, r in results.items():
        print(f"  {name:35s} top-20% segments: {r['segments_top20']}")

    # ---- 5. Charts ----
    main_names = [chosen, "Churn risk (Random Forest)", "Random"]
    colors = {chosen: "#55A868", "Churn risk (Random Forest)": "#DD8452", "Random": "#8C8C8C"}

    # 5a. Qini curves (observed data: what a real company could measure)
    plt.figure(figsize=(7, 5))
    qini_points = {}
    for name in main_names:
        x, yq = qini_curve(y_test, scores[name], t_test)
        plt.plot(x / len(test) * 100, yq, label=f"{name} (Qini AUC={results[name]['qini_auc']:.3f})",
                 color=colors[name])
        idx = np.linspace(0, len(x) - 1, 21).astype(int)
        qini_points[name] = [{"pct": round(float(x[i] / len(test) * 100), 1), "qini": round(float(yq[i]), 2)} for i in idx]
    plt.axvline(TOP_K * 100, ls=":", color="black", label="Top 20% budget")
    plt.xlabel("% of customers contacted (ranked by score)")
    plt.ylabel("Incremental customers retained (Qini)")
    plt.title("Qini curves on test set (SIMULATED offer data)")
    plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIG_DIR / "qini_curves.png", dpi=120); plt.close()

    # 5b. True gain curves (ground truth: only possible in a simulation)
    plt.figure(figsize=(7, 5))
    gain_points = {}
    for name in main_names:
        fr, g = true_gain_curve(scores[name], true_effect)
        plt.plot(fr * 100, g, marker="o", ms=3, label=name, color=colors[name])
        gain_points[name] = [{"pct": round(float(a * 100)), "saved": round(float(b), 1)} for a, b in zip(fr, g)]
    plt.axvline(TOP_K * 100, ls=":", color="black", label="Top 20% budget")
    plt.xlabel("% of customers contacted"); plt.ylabel("Expected customers saved (true, simulated)")
    plt.title("Customers saved vs % contacted (ground truth)")
    plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIG_DIR / "uplift_gain.png", dpi=120); plt.close()

    # 5c. Observed uplift by decile of the chosen uplift model
    dec = pd.DataFrame({"score": scores[chosen], "y": y_test.values, "t": t_test.values})
    dec["decile"] = pd.qcut(-dec["score"].rank(method="first"), 10, labels=range(1, 11))
    by_dec = dec.groupby("decile", observed=True).apply(
        lambda g: g.loc[g.t == 1, "y"].mean() - g.loc[g.t == 0, "y"].mean(), include_groups=False) * 100
    plt.figure(figsize=(7, 4))
    plt.bar(by_dec.index.astype(str), by_dec.values,
            color=["#55A868" if v > 0 else "#C44E52" for v in by_dec.values])
    plt.axhline(0, color="black", lw=0.8)
    plt.xlabel("Decile of predicted uplift (1 = highest)")
    plt.ylabel("Observed uplift (pts)\nstay% offer − stay% no offer")
    plt.title(f"Observed uplift by decile: {chosen}")
    plt.tight_layout(); plt.savefig(FIG_DIR / "uplift_by_decile.png", dpi=120); plt.close()

    # ---- 6. Save results JSON ----
    with open(RESULTS_DIR / "uplift_metrics.json", "w") as f:
        json.dump({
            "simulated": True,
            "note": "Treatment and its effect are SIMULATED (see ml/simulate_treatment.py).",
            "chosen_uplift_model": chosen,
            "cv_qini_train": {n: round(v, 4) for n, v in cv_scores.items()},
            "test_size": len(test), "top_k": TOP_K, "k_customers": k,
            "best_possible_saved_top20": round(float(best_possible), 1),
            "methods": results,
            "qini_curves": qini_points,
            "gain_curves": gain_points,
            "uplift_by_decile": {str(i): round(float(v), 2) for i, v in by_dec.items()},
        }, f, indent=2)
    print(f"\nSaved uplift model + results to {MODEL_DIR} and {RESULTS_DIR}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/score_customers.py
"""
Build the customer file for "myDel", a FICTIONAL phone company used in the demo app.

myDel's customers = all 7,043 real (anonymous) records from the IBM Telco dataset,
with generated FAKE names / regions so the demo feels like a real customer system.

For every customer we add:
  churn_prob     chance of leaving    (Random Forest, calibrated)
  uplift_score   how much a discount raises their chance of staying (T-learner, SIMULATED offer data)
  reasons_leave / reasons_stay   top 3 reasons in plain English (from SHAP)

Honest scoring: OUT-OF-FOLD predictions. The customers are split into 5 groups, and
each group is scored by models trained on the OTHER 4 groups. So no customer is
scored by a model that saw them during training.

Also writes ml/results/mydel_roi.json: campaign profit for myDel's whole customer base.

Run from the project root (after train_classical.py and train_uplift.py):
    python ml/score_customers.py
"""
import json
from datetime import date

import numpy as np
import pandas as pd
import shap
import joblib
import yaml
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

from preprocess import FEATURE_COLS, ID_COL, PROJECT_ROOT, RANDOM_STATE, build_preprocessor, load_clean
from simulate_treatment import simulate_treatment
from train_uplift import TLearner

OUT_PATH = PROJECT_ROOT / "backend" / "data" / "mydel_customers.csv"
ROI_PATH = PROJECT_ROOT / "ml" / "results" / "mydel_roi.json"
TODAY = date(2026, 11, 1)  # fixed "today" so the file is reproducible

# ---------- fake names (fictional demo data) ----------
FIRST_F = ["Aisha", "Emma", "Priya", "Sofia", "Mia", "Olivia", "Zara", "Lena", "Hana", "Chloe", "Nora", "Isla",
           "Maya", "Ava", "Leila", "Grace", "Anika", "Ruby", "Elena", "Sara", "Ivy", "Meera", "Lucy", "Amara",
           "Clara", "Nina", "Yara", "Ella", "Tara", "Rosa"]
FIRST_M = ["Arjun", "Liam", "Omar", "Noah", "Ethan", "Ravi", "Lucas", "Mateo", "Kai", "Leo", "Sami", "Daniel",
           "Aditya", "James", "Felix", "Hugo", "Ibrahim", "Oscar", "Rohan", "Theo", "Adam", "Marco", "Kabir",
           "Jonas", "Elias", "Dev", "Max", "Ali", "Nikhil", "Ben"]
LAST = ["Sharma", "Smith", "Khan", "Garcia", "Chen", "Patel", "Müller", "Rossi", "Kim", "Silva", "Novak", "Haddad",
        "Brown", "Singh", "Lopez", "Ivanova", "Okafor", "Tanaka", "Wilson", "Dubois", "Mehta", "Costa", "Larsen",
        "Ahmed", "Fischer", "Reddy", "Nguyen", "Martin", "Kowalski", "Rahman", "Taylor", "Bose", "Moreau",
        "Santos", "Jensen", "Iyer", "Clarke", "Yilmaz", "Kapoor", "Evans"]
REGIONS = ["North", "South", "East", "West", "Central"]

# ---------- plain-English description of a customer's value for each column ----------
YES_NO = {
    "Partner": ("Has a partner", "No partner"),
    "Dependents": ("Has children / dependents", "No dependents"),
    "PhoneService": ("Has a phone line", "No phone line"),
    "MultipleLines": ("Multiple phone lines", "Single phone line"),
    "OnlineSecurity": ("Has online security", "No online security"),
    "OnlineBackup": ("Has online backup", "No online backup"),
    "DeviceProtection": ("Has device protection", "No device protection"),
    "TechSupport": ("Has tech support", "No tech support"),
    "StreamingTV": ("Streams TV", "No TV streaming"),
    "StreamingMovies": ("Streams movies", "No movie streaming"),
    "PaperlessBilling": ("Paperless billing", "Gets paper bills"),
}
CATEGORY = {
    "Contract": {"Month-to-month": "Month-to-month contract (can leave any time)",
                 "One year": "On a 1-year contract", "Two year": "On a 2-year contract"},
    "InternetService": {"Fiber optic": "Fiber internet (pricier plan)", "DSL": "DSL internet", "No": "No internet service"},
    "PaymentMethod": {"Electronic check": "Pays by electronic check", "Mailed check": "Pays by mailed check",
                      "Bank transfer (automatic)": "Automatic bank payments",
                      "Credit card (automatic)": "Automatic card payments"},
}


def describe(col: str, row: pd.Series) -> str:
    v = row[col]
    if col == "tenure":
        months = f"{v} month" + ("" if v == 1 else "s")
        return f"New customer (only {months} with myDel)" if v < 12 else f"Loyal for {months}"
    if col == "MonthlyCharges":
        return f"{'High' if v >= 70 else 'Low' if v < 35 else 'Medium'} monthly bill (${v:.0f})"
    if col == "TotalCharges":
        return f"Has spent only ${v:,.0f} with myDel so far" if v < 1000 else f"Has spent ${v:,.0f} with myDel"
    if col == "SeniorCitizen":
        return "Senior citizen" if v == 1 else "Not a senior citizen"
    if col == "gender":
        return str(v)
    if col in YES_NO:
        return YES_NO[col][0] if v == "Yes" else YES_NO[col][1]
    return CATEGORY.get(col, {}).get(v, f"{col}: {v}")


def column_of(feature: str) -> str:
    """'cat__Contract_Two year' -> 'Contract' (map one-hot feature back to its column)."""
    name = feature.split("__", 1)[1]
    for col in FEATURE_COLS:
        if name == col or name.startswith(col + "_"):
            return col
    return name


def main() -> None:
    rng = np.random.default_rng(RANDOM_STATE)
    sim = simulate_treatment(load_clean()).reset_index(drop=True)
    X, y = sim[FEATURE_COLS], sim["Churn"]

    # ---- 1. Out-of-fold scores (each customer scored by models that never saw them) ----
    rf = Pipeline([("preprocess", build_preprocessor()),
                   ("model", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5,
                                                    class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))])
    rf = CalibratedClassifierCV(rf, method="sigmoid", cv=5)  # same calibration as train_classical.py
    churn_prob = np.zeros(len(sim))
    uplift = np.zeros(len(sim))
    folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    for i, (tr, te) in enumerate(folds.split(X, sim["treatment"] * 2 + y)):
        churn_prob[te] = rf.fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
        t = TLearner(LogisticRegression(max_iter=1000)).fit(
            X.iloc[tr], 1 - sim["churn_sim"].iloc[tr], sim["treatment"].iloc[tr])
        uplift[te] = t.predict_uplift(X.iloc[te])
        print(f"  fold {i + 1}/5 scored")
    sim["churn_prob"], sim["uplift_score"] = churn_prob, uplift

    # ---- 2. Plain-English reasons per customer (SHAP on the saved churn model) ----
    # SHAP needs the plain tree model; calibration rescales scores but doesn't change which features matter
    model = joblib.load(PROJECT_ROOT / "ml" / "models" / "churn_model_uncalibrated.joblib")
    pre, clf = model.named_steps["preprocess"], model.named_steps["model"]
    print("  computing SHAP reasons (about a minute)...")
    sv = shap.TreeExplainer(clf)(pre.transform(X)).values
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    # add up one-hot pieces so each ORIGINAL column has one contribution
    cols = [column_of(f) for f in pre.get_feature_names_out()]
    contrib = pd.DataFrame(sv, columns=cols).T.groupby(level=0).sum().T
    leave, stay = [], []
    for i in range(len(sim)):
        c = contrib.iloc[i].drop("gender").sort_values()  # gender isn't a useful "reason"
        row = sim.iloc[i]
        # biggest positive contributions = reasons to LEAVE; biggest negative = reasons to STAY
        leave.append(" | ".join([describe(k, row) for k, v in c[::-1].items() if v > 0.005][:3]))
        stay.append(" | ".join([describe(k, row) for k, v in c.items() if v < -0.005][:3]))
    sim["reasons_leave"], sim["reasons_stay"] = leave, stay

    # ---- 3. Fictional profile fields ----
    first = [rng.choice(FIRST_F if g == "Female" else FIRST_M) for g in sim["gender"]]
    sim["name"] = [f"{f} {rng.choice(LAST)}" for f in first]
    sim["region"] = rng.choice(REGIONS, len(sim))
    sim["customer_since"] = [
        (pd.Timestamp(TODAY) - pd.DateOffset(months=int(t))).strftime("%b %Y") for t in sim["tenure"]]

    keep = [ID_COL, "name", "region", "customer_since"] + FEATURE_COLS + \
           ["Churn", "churn_prob", "uplift_score", "reasons_leave", "reasons_stay"]
    out = sim[keep].copy()
    out[["churn_prob", "uplift_score"]] = out[["churn_prob", "uplift_score"]].round(4)
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(OUT_PATH, index=False)

    # ---- 4. Campaign profit for ALL myDel customers (same method as business_roi.py) ----
    with open(PROJECT_ROOT / "params.yaml") as f:
        biz = yaml.safe_load(f)["business"]
    value = sim["MonthlyCharges"] * biz["clv_months"] * sim["true_effect"]  # $ saved if contacted (simulated truth)
    rankings = {"Churn risk": churn_prob, "Uplift (T-learner)": uplift, "Random": rng.random(len(sim))}
    table = []
    for method, score in rankings.items():
        order = np.argsort(-score)
        for pct in biz["target_percents"]:
            n = int(round(pct / 100 * len(sim)))
            cost = n * biz["offer_cost"]
            gain = float(value.to_numpy()[order[:n]].sum())
            table.append({"method": method, "percent": pct, "contacted": n,
                          "customers_saved": round(float(sim["true_effect"].to_numpy()[order[:n]].sum()), 1),
                          "cost": cost, "value_saved": round(gain), "profit": round(gain - cost),
                          "roi": round((gain - cost) / cost, 3)})
    with open(ROI_PATH, "w") as f:
        json.dump({"simulated": True, "customers": len(sim),
                   "assumptions": {"offer_cost": biz["offer_cost"], "clv_months": biz["clv_months"]},
                   "table": table}, f, indent=2)

    print(f"\nSaved {len(out):,} myDel customers -> {OUT_PATH}")
    print(f"Saved campaign profit table -> {ROI_PATH}")
    print(pd.DataFrame(table).pivot(index="percent", columns="method", values="profit").to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile params.yaml
# Central configuration for the project.
# Business values are ASSUMPTIONS (demo values). Change them here and re-run:
#   python ml/business_roi.py

business:
  offer_cost: 50            # $ cost of one retention offer (e.g. one-month discount), paid for EVERY contacted customer
  clv_months: 12            # value of keeping a customer = their MonthlyCharges x this many months
  target_percents: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]   # campaign sizes to compare
  headline_percent: 20      # "we can only contact the top 20%"

decision:                   # rules used by POST /predict to recommend an action
  low_risk_threshold: 0.30  # churn probability below this -> "no action needed"
  backfire_margin: 0.02     # predicted uplift below MINUS this -> "do not contact".
                            # Uplift within +/-0.02 of zero is model noise, not a real backfire.


## Setup 3 · Download the data and run the pipeline

- `ml/download_data.py`
- `ml/preprocess.py`
- `ml/train_classical.py`
- `ml/train_uplift.py`
- `ml/score_customers.py`

Outputs are hidden unless a step fails.

In [ ]:
import subprocess, sys, time
for script in ['ml/download_data.py', 'ml/preprocess.py', 'ml/train_classical.py', 'ml/train_uplift.py', 'ml/score_customers.py']:
    t = time.time()
    r = subprocess.run([sys.executable, script], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"{script} failed")
    print(f"✓ {script}  ({time.time() - t:.0f} s)")
print("Setup complete.")

# 02 · Modelling: churn prediction, uplift, and the business case

**Group 11 · Track 3 Project Showcase · Topic A: Customer Churn Prediction with Simple Uplift Extension**

This notebook is the end-to-end ML case study. It follows on from `01_eda.ipynb` (data exploration) and runs the real pipeline:

1. Split the data the same way every time
2. Train and compare three **classical churn models**
3. **Calibrate** the best model so its probabilities are honest
4. Explain it with **SHAP**
5. Build a **simulated** retention-offer trial (the dataset has none)
6. Train a **T-learner uplift model**
7. Compare targeting by **churn risk** vs **uplift** vs **random**
8. Turn scores into **profit** and a **business recommendation**
9. Show the **decision rules** the demo app uses
10. Check that every number matches the saved results the app reads

> **How this notebook relates to the code.** It imports the project's own functions from `ml/` (`preprocess.py`, `train_classical.py`, `simulate_treatment.py`, `train_uplift.py`), so the logic is identical to the scripts. It trains everything **in memory** and does **not** overwrite the saved models in `ml/models/`.
>
> ⚠️ **Sections 5 onward use a simulated discount trial.** The IBM Telco data never recorded who received an offer. All uplift and dollar figures illustrate the method; they are not real campaign results.

## 0. Setup

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import yaml
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (ConfusionMatrixDisplay, brier_score_loss, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklift.metrics import qini_auc_score, qini_curve

warnings.filterwarnings("ignore")

# Import the project's own code from ml/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "ml"))
from preprocess import FEATURE_COLS, RANDOM_STATE, TARGET, build_preprocessor, load_clean, split_data
from train_classical import THRESHOLD, evaluate, make_models, pretty_name
from simulate_treatment import SAVE_RATE, BACKFIRE_RATE, simulate_treatment
from train_uplift import TOP_K, TLearner, base_models, cv_qini, true_gain_curve
%matplotlib inline
# (the ml/ scripts switch matplotlib to file-only mode; this turns inline charts back on)

RESULTS = PROJECT_ROOT / "ml" / "results"
PARAMS = yaml.safe_load(open(PROJECT_ROOT / "params.yaml"))
sns.set_theme(style="whitegrid")
COLORS = {"Logistic Regression": "#8C8C8C", "Random Forest": "#C44E52", "XGBoost": "#4C72B0",
          "Churn risk": "#C44E52", "Uplift (T-learner)": "#8172B3", "Random": "#8C8C8C"}
print("Project root:", PROJECT_ROOT)

## 1. Data and the train/test split

We use the cleaned data from `01_eda.ipynb` / `ml/preprocess.py` (blank `TotalCharges` set to 0, "No internet service" merged into "No", `Churn` as 1/0).

The split is **80% train / 20% test, stratified on churn** with a fixed seed. Every model in this notebook is judged on the **same 1,409 test customers**, which none of them see during training.

In [ ]:
df = load_clean()
train, test = split_data(df)
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]

print(f"All customers: {len(df):,}   features: {len(FEATURE_COLS)}")
print(f"Train: {len(train):,}  churn rate {y_train.mean():.1%}")
print(f"Test:  {len(test):,}  churn rate {y_test.mean():.1%}")

The data is **imbalanced**: about 3 stayers for every leaver. A model that predicts "everyone stays" would be 73% accurate and completely useless, so we don't use accuracy as the headline metric.

## 2. Three classical churn models

| Model | Idea | Why include it |
|---|---|---|
| **Logistic Regression** | weighted sum of features → probability | simple, explainable baseline |
| **Random Forest** | 300 decision trees vote | captures interactions, robust |
| **XGBoost** | trees built one after another, each fixing the last one's errors | usually strongest on tabular data |

Each model sits inside a scikit-learn **Pipeline**: `[one-hot encode + scale] → [model]`. The encoder and scaler learn only from the training rows, so nothing leaks from the test set.

**Class imbalance** is handled with class weights (`class_weight="balanced"`, or `scale_pos_weight` for XGBoost): a missed leaver costs about 3× more during training.

**Metrics** (test set, threshold 0.5):
- **ROC-AUC**: how well the model *ranks* leavers above stayers (threshold-free; 0.5 = random, 1 = perfect)
- **Recall**: of customers who really left, the share we flagged. This matters most for retention, because a missed leaver is lost revenue
- **Precision**: of customers we flagged, the share who really left
- **5-fold CV AUC** on the training data, to check the result is stable and not luck

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results, pipelines, probs = {}, {}, {}
for name, model in make_models(pos_weight).items():
    pipe = Pipeline([("preprocess", build_preprocessor()), ("model", model)])
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    m = evaluate(y_test, prob)
    m["cv_auc_mean"], m["cv_auc_std"] = cv_auc.mean(), cv_auc.std()
    results[name], pipelines[name], probs[name] = m, pipe, prob

leaderboard = pd.DataFrame(results).T[["roc_auc", "precision", "recall", "f1", "cv_auc_mean", "cv_auc_std"]]
leaderboard.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

leaderboard[["roc_auc", "precision", "recall", "f1"]].T.plot.bar(
    ax=axes[0], color=[COLORS[n] for n in leaderboard.index], rot=0)
axes[0].set_title("Test-set metrics by model"); axes[0].set_ylim(0, 1); axes[0].legend(loc="lower right")

for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[1].plot(fpr, tpr, color=COLORS[name], label=f"{name} (AUC {results[name]['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random guess (AUC 0.5)")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate (recall)", title="ROC curves (test set)")
axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
best_name = max(results, key=lambda n: results[n]["roc_auc"])
best_pipe = pipelines[best_name]
spread = leaderboard["roc_auc"].max() - leaderboard["roc_auc"].min()
print(f"Best model by test ROC-AUC: {best_name} ({results[best_name]['roc_auc']:.3f})")
print(f"Spread between best and worst model: {spread:.3f} AUC")
print(f"CV standard deviation of the best model: {results[best_name]['cv_auc_std']:.3f}")

**Reading the result.** Random Forest has the highest test ROC-AUC, but all three models are within about 0.004 of each other, which is smaller than the cross-validation spread. The honest conclusion: **the information in the data, not the choice of algorithm, limits performance.** We keep Random Forest (best AUC and best recall) and don't spend effort on heavy tuning.

### Confusion matrix of the chosen model

In [ ]:
cm = confusion_matrix(y_test, (probs[best_name] >= THRESHOLD).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["Stayed", "Churned"]).plot(cmap="Blues", colorbar=False)
plt.title(f"{best_name}, threshold {THRESHOLD}"); plt.grid(False); plt.show()
tn, fp, fn, tp = cm.ravel()
print(f"Leavers caught: {tp}   missed: {fn}   false alarms: {fp}   stayers correctly left alone: {tn}")

The model catches about 4 in 5 leavers at the cost of some false alarms. For retention that trade-off is right: a false alarm costs at most one offer, a missed leaver costs a year of revenue.

## 3. Calibration: making the probabilities honest

Class weights help the model find leavers, but they inflate its probabilities. The demo app shows each customer's **chance of staying**, so the number must mean what it says.

We wrap the chosen pipeline in `CalibratedClassifierCV` with **Platt scaling** (`method="sigmoid"`, 5 folds on the training data). It re-maps scores with a small logistic curve: **the ranking is unchanged** (same AUC), only the scale is corrected.

In [ ]:
calibrated = CalibratedClassifierCV(clone(best_pipe), method="sigmoid", cv=5).fit(X_train, y_train)
raw_prob, cal_prob = probs[best_name], calibrated.predict_proba(X_test)[:, 1]

cal_table = pd.DataFrame({
    "mean predicted churn": [raw_prob.mean(), cal_prob.mean()],
    "actual churn rate": [y_test.mean()] * 2,
    "Brier score (lower = better)": [brier_score_loss(y_test, raw_prob), brier_score_loss(y_test, cal_prob)],
    "ROC-AUC": [roc_auc_score(y_test, raw_prob), roc_auc_score(y_test, cal_prob)],
}, index=["before (class-weighted)", "after calibration"])
display(cal_table.round(4))

plt.figure(figsize=(5.5, 5))
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Perfectly calibrated")
for label, p, c in [("Before", raw_prob, "#DD8452"), ("After", cal_prob, "#4C72B0")]:
    frac, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac, "o-", color=c, label=label)
plt.xlabel("Predicted churn probability"); plt.ylabel("Actual churn rate")
plt.title("Calibration curve (test set)"); plt.legend(); plt.show()

Before calibration the model predicted ~39% churn on average against a real 26.5%. After Platt scaling the average matches reality, the Brier score improves, and ROC-AUC is unchanged. **The calibrated model is the one the app uses.**

## 4. Explaining the model with SHAP

SHAP splits every prediction into **one contribution per feature** (for example, "month-to-month contract pushed this customer +12 points towards leaving"). Averaging the size of those contributions gives global importance; each dot in the beeswarm is one test customer.

We explain the uncalibrated Random Forest: calibration rescales the output but doesn't change which features drive it.

In [ ]:
pre, rf = best_pipe.named_steps["preprocess"], best_pipe.named_steps["model"]
X_test_t = pre.transform(X_test)
sv = shap.TreeExplainer(rf)(X_test_t)
sv.feature_names = [pretty_name(f) for f in pre.get_feature_names_out()]
if sv.values.ndim == 3:          # Random Forest returns one set per class: keep "churn"
    sv = sv[:, :, 1]
shap.plots.beeswarm(sv, max_display=12, show=False)
plt.title("SHAP: what pushes a customer towards churn (right) or staying (left)"); plt.show()

In [ ]:
mean_abs = np.abs(sv.values).mean(axis=0)
drivers = pd.DataFrame({"feature": sv.feature_names, "mean |SHAP|": mean_abs})
drivers["direction"] = [
    "increases churn" if np.corrcoef(sv.data[:, i], sv.values[:, i])[0, 1] > 0 else "decreases churn"
    for i in range(len(drivers))]
drivers.sort_values("mean |SHAP|", ascending=False).head(10).round(4).reset_index(drop=True)

The drivers agree with the EDA: **month-to-month contracts, fiber internet and short tenure** push customers out; **long tenure and two-year contracts** keep them. In the app, each customer's three biggest positive and negative SHAP contributions become the plain-English *"Why they might leave / stay"* lists.

## 5. Why a churn model isn't enough: uplift

A churn model answers *who will leave*. A retention budget needs *whose decision we can change*. Every customer falls into one of four groups:

| | Would **leave** without an offer | Would **stay** without an offer |
|---|---|---|
| **Stays** with an offer | **Persuadables**: the only group the discount works on | **Sure things**: the discount is wasted |
| **Leaves** with an offer | **Lost causes**: the discount is wasted | **Sleeping dogs**: contacting them backfires |

Ranking by churn risk puts persuadables *and* lost causes at the top. An **uplift model** estimates, per customer, how much the offer changes the chance of staying, so it can tell them apart.

### 5.1 The simulated retention trial ⚠️

Uplift needs data on who got an offer. Telco has none, so `ml/simulate_treatment.py` creates a **randomised trial**:

1. Each customer gets the offer with probability 50% (like a fair A/B test).
2. Customers **without** the offer keep their real outcome.
3. Customers **with** the offer may change outcome according to **assumed** rules per segment:

In [ ]:
assumptions = pd.DataFrame({
    "rule": ["Two-year contract AND tenure > 48 months",
             "Fiber AND electronic check AND bill > $90",
             "Month-to-month AND tenure ≤ 12 months",
             "Any other month-to-month",
             "Any other 1- or 2-year contract"],
    "assumed save rate (of real leavers)": [SAVE_RATE[s] for s in ["sleeping_dog", "lost_cause", "persuadable", "m2m_other", "contract"]],
    "assumed backfire rate (of real stayers)": [BACKFIRE_RATE.get(s, 0.0) for s in ["sleeping_dog", "lost_cause", "persuadable", "m2m_other", "contract"]],
}, index=["sleeping_dog", "lost_cause", "persuadable", "m2m_other", "contract"])
assumptions

In [ ]:
sim = simulate_treatment(df)
g = sim.groupby("segment")
seg = pd.DataFrame({
    "customers": g.size(),
    "real churn rate": g["Churn"].mean(),
    "churn without offer": sim[sim.treatment == 0].groupby("segment")["churn_sim"].mean(),
    "churn with offer": sim[sim.treatment == 1].groupby("segment")["churn_sim"].mean(),
})
seg["observed effect (pts)"] = (seg["churn without offer"] - seg["churn with offer"]) * 100
display(seg.loc[["persuadable", "m2m_other", "lost_cause", "contract", "sleeping_dog"]].round(3))

ctrl, trt = sim.loc[sim.treatment == 0, "churn_sim"].mean(), sim.loc[sim.treatment == 1, "churn_sim"].mean()
print(f"Treated {sim.treatment.sum():,} / control {(1 - sim.treatment).sum():,}")
print(f"Overall churn without offer {ctrl:.1%}, with offer {trt:.1%} -> average effect {100 * (ctrl - trt):.1f} points")

Because **we** wrote the rules, we also know each customer's **true effect** (`true_effect`: expected customers saved by contacting them). A real company never has this. Here it lets us score every targeting strategy exactly. It is used **only for evaluation**, never as a model input.

## 6. The T-learner uplift model

**T-learner = two models:**
- **Model T** learns from customers who **got** the offer → P(stay | offer)
- **Model C** learns from customers who **didn't** → P(stay | no offer)
- **Uplift = P(stay | offer) − P(stay | no offer)**: positive helps, ~0 no effect, negative backfires

We try two base learners and choose between them with **5-fold cross-validated Qini on the training data only**. Qini is the standard uplift metric: how many *extra* customers stay as you contact more people in score order.

In [ ]:
sim_train, sim_test = split_data(sim)          # same customers as Section 1
Xs_train, Xs_test = sim_train[FEATURE_COLS], sim_test[FEATURE_COLS]
ys_train, ys_test = 1 - sim_train["churn_sim"], 1 - sim_test["churn_sim"]   # 1 = stayed
t_train, t_test = sim_train["treatment"], sim_test["treatment"]

cv_scores = {name: cv_qini(base, Xs_train, ys_train, t_train) for name, base in base_models().items()}
chosen = max(cv_scores, key=cv_scores.get)
display(pd.Series(cv_scores, name="5-fold CV Qini (train)").round(4).to_frame())
print("Chosen base learner:", chosen)

Logistic Regression wins: the uplift signal is small and noisy (it's a *difference* of two predictions), and the simpler model generalises better than XGBoost here.

## 7. Which ranking saves more customers?

The question from the brief: **classical churn model vs basic uplift.** With a budget to contact the **top 20%** of test customers, we rank them three ways and count the customers each ranking truly saves:

- **Churn risk**: the calibrated Random Forest from Section 3
- **Uplift**: the chosen T-learner
- **Random**: the baseline any model must beat

In [ ]:
tlearner = TLearner(base_models()[chosen]).fit(Xs_train, ys_train, t_train)
rankings = {
    "Churn risk": calibrated.predict_proba(Xs_test)[:, 1],
    "Uplift (T-learner)": tlearner.predict_uplift(Xs_test),
    "Random": np.random.default_rng(RANDOM_STATE).random(len(sim_test)),
}
true_effect = sim_test["true_effect"].to_numpy()
k = int(round(TOP_K * len(sim_test)))

rows = {}
for name, s in rankings.items():
    top = np.argsort(-s)[:k]
    segs = sim_test.iloc[top]["segment"].value_counts()
    rows[name] = {
        "customers truly saved (top 20%)": round(true_effect[top].sum(), 1),
        "Qini AUC": qini_auc_score(ys_test, s, t_test),
        "persuadables contacted": int(segs.get("persuadable", 0)),
        "lost causes contacted": int(segs.get("lost_cause", 0)),
        "sleeping dogs contacted": int(segs.get("sleeping_dog", 0)),
    }
comparison = pd.DataFrame(rows).T
count_cols = ["persuadables contacted", "lost causes contacted", "sleeping dogs contacted"]
comparison[count_cols] = comparison[count_cols].astype(int)
best_possible = np.sort(true_effect)[::-1][:k].sum()
display(comparison.round(3))
print(f"Top 20% = {k} customers. A perfect ranking would save {best_possible:.1f}.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for name, s in rankings.items():
    x, yq = qini_curve(ys_test, s, t_test)
    axes[0].plot(x / len(sim_test) * 100, yq, color=COLORS[name], label=name)
    fr, gain = true_gain_curve(s, true_effect)
    axes[1].plot(fr * 100, gain, "o-", ms=3, color=COLORS[name], label=name)
for ax, title, ylabel in [(axes[0], "Qini curve (what a company could measure)", "Incremental customers retained"),
                          (axes[1], "True customers saved (simulation ground truth)", "Customers saved")]:
    ax.axvline(TOP_K * 100, ls=":", color="black", lw=1, label="20% budget")
    ax.set(xlabel="% of test customers contacted", ylabel=ylabel, title=title); ax.legend()
plt.tight_layout(); plt.show()

**Result.** Both models beat random by roughly **3×**. **Churn-risk ranking saves the most customers** at the 20% budget, with uplift close behind.

**Why didn't uplift win?** In this simulation the persuadables (new month-to-month customers) are *also* the highest-risk customers, so ranking by risk already finds most of them. Uplift does what it promises: it contacts **far fewer lost causes**. But its estimates are noisier, because they are the difference between two models trained on half the data each. A Logistic Regression T-learner also can't learn the AND-rules that define the lost-cause segment.

This conclusion depends on the simulation's assumptions; with real trial data the result could differ.

## 8. From scores to money

**Business assumptions** (`params.yaml`; demo values, easy to change):
- each offer costs **${offer}**, paid for every contacted customer
- keeping a customer is worth **{months} months** of their bill

For each ranking and each campaign size (10%, 20% … 100% contacted):

    cost         = customers contacted × offer cost
    value saved  = Σ true_effect × monthly bill × 12   (over contacted customers)
    profit       = value saved − cost

In [ ]:
biz = PARAMS["business"]
customer_value = (sim_test["MonthlyCharges"] * biz["clv_months"]).to_numpy()
dollar_effect = true_effect * customer_value

roi_rows = []
for name, s in rankings.items():
    order = np.argsort(-s)
    for pct in biz["target_percents"]:
        n = int(round(pct / 100 * len(sim_test)))
        cost = n * biz["offer_cost"]
        value = dollar_effect[order[:n]].sum()
        roi_rows.append({"method": name, "percent": pct, "contacted": n, "profit": value - cost,
                         "ROI": (value - cost) / cost})
roi_test = pd.DataFrame(roi_rows)
roi_test.pivot(index="percent", columns="method", values="profit").round(0)

### The whole customer base (what the demo app shows)

The app scores **all 7,043 customers** with **out-of-fold** predictions (`ml/score_customers.py`): customers are split into 5 groups and each group is scored by models trained on the other 4, so no customer is scored by a model that saw them. That run takes a few minutes, so we load its saved output here.

In [ ]:
mydel = pd.DataFrame(json.load(open(RESULTS / "mydel_roi.json"))["table"])
curve = mydel.pivot(index="percent", columns="method", values="profit")

plt.figure(figsize=(8, 4.6))
for name in ["Churn risk", "Uplift (T-learner)", "Random"]:
    plt.plot(curve.index, curve[name] / 1000, "o-", color=COLORS[name], label=name,
             ls="--" if name == "Random" else "-")
peak = mydel.loc[mydel["profit"].idxmax()]
plt.scatter([peak.percent], [peak.profit / 1000], s=90, color="black", zorder=5)
plt.annotate(f"Peak ${peak.profit / 1000:,.0f}k at {peak.percent}%", (peak.percent, peak.profit / 1000),
             textcoords="offset points", xytext=(12, -16), fontweight="bold")
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("% of customers contacted"); plt.ylabel("Profit ($ thousands)")
plt.title("Campaign profit, all 7,043 myDel customers (SIMULATED offer effect)"); plt.legend(); plt.show()

peaks = mydel.loc[mydel.groupby("method")["profit"].idxmax(), ["method", "percent", "contacted", "profit", "roi"]]
peaks.set_index("method").round(2)

### Business recommendation

1. **Target by churn risk and contact the top ~40%** (≈2,800 customers): about **$207k** profit over a year in this simulation, versus about $183k for uplift ranking and almost nothing for random picks.
2. **Don't contact everyone.** Past ~40% the extra offers mostly go to sure things, and profit falls to roughly zero at 100%.
3. **Use the uplift model as a filter, not the ranking:** skip customers it flags as sleeping dogs (negative uplift).
4. **Run a real A/B test before trusting uplift scores**, because the offer effects here are simulated.

## 9. The decision rules in the demo app

`backend/app/model_service.py` turns the two scores into one of four actions per customer, checked in this order (thresholds from `params.yaml`):

1. uplift < −0.02 → **Don't contact** (sleeping dog; tiny negatives are treated as noise)
2. churn probability < 30% → **No action needed** (they'll probably stay)
3. expected profit = uplift × customer value − $50 > 0 → **Send the discount**
4. otherwise → **Call them instead** (high risk, but money isn't the fix)

In [ ]:
dec = PARAMS["decision"]

def decide(churn_prob, uplift, customer_value):
    if uplift < -dec["backfire_margin"]:
        return "Do not contact"
    if churn_prob < dec["low_risk_threshold"]:
        return "No action needed"
    if uplift * customer_value - biz["offer_cost"] > 0:
        return "Send retention offer"
    return "Try non-discount retention"

customers = pd.read_csv(PROJECT_ROOT / "backend" / "data" / "mydel_customers.csv")
customers["action"] = [decide(p, u, m * biz["clv_months"]) for p, u, m in
                       zip(customers["churn_prob"], customers["uplift_score"], customers["MonthlyCharges"])]
customers["action"].value_counts().to_frame("customers")

## 10. Reproducibility check

Everything above was retrained from scratch in this notebook. These checks confirm it matches the saved results in `ml/results/`, which are the numbers the demo app displays.

In [ ]:
saved_cls = json.load(open(RESULTS / "classical_metrics.json"))
saved_upl = json.load(open(RESULTS / "uplift_metrics.json"))

checks = []
for name in results:
    checks.append((f"{name} ROC-AUC", results[name]["roc_auc"], saved_cls["models"][name]["roc_auc"]))
checks.append(("Best model", best_name, saved_cls["best_model"]))
checks.append(("Calibrated mean churn", cal_prob.mean(), saved_cls["calibration"]["after"]["mean_predicted"]))
checks.append(("Chosen uplift learner", chosen, saved_upl["chosen_uplift_model"]))
checks.append(("Uplift: saved in top 20%", rows["Uplift (T-learner)"]["customers truly saved (top 20%)"],
               saved_upl["methods"][chosen]["true_saved_top20"]))
checks.append(("Churn risk: saved in top 20%", rows["Churn risk"]["customers truly saved (top 20%)"],
               saved_upl["methods"]["Churn risk (Random Forest)"]["true_saved_top20"]))
checks.append(("Best possible in top 20%", round(best_possible, 1), saved_upl["best_possible_saved_top20"]))

def same(a, b):
    return a == b if isinstance(a, str) else abs(float(a) - float(b)) < 1e-3 + 1e-3 * abs(float(b))

report = pd.DataFrame([(n, a, b, "✓" if same(a, b) else "✗") for n, a, b in checks],
                      columns=["check", "this notebook", "saved results", "match"])
display(report)
assert report["match"].eq("✓").all(), "Some results differ from ml/results/"
print("All checks pass: the notebook reproduces the saved results.")

## 11. Conclusions and limitations

**What we found**
- Three classical models reach **ROC-AUC ≈ 0.84–0.85** and recall ≈ 0.79. Random Forest is chosen, but the models are statistically tied; the data limits performance.
- **Calibration** fixed inflated probabilities (≈39% → 26.7% average predicted churn) without changing the ranking.
- **SHAP** confirms the EDA: contract type, tenure and fiber internet drive churn.
- With a 20% budget, **churn-risk ranking saved the most customers**; the **T-learner uplift model** came close and wasted far fewer offers on lost causes. Both beat random by ~3×.
- **Recommendation:** contact the top ~40% by churn risk, filter out predicted sleeping dogs, and validate with a real A/B test.

**Limitations**
- ⚠️ The offer and its effects are **simulated**; all uplift and dollar results demonstrate the method, not real customer behaviour.
- The simulation's assumptions favour churn-risk ranking (the offer mainly helps real leavers), so *"risk beats uplift"* partly follows from the design.
- A Logistic Regression T-learner can't learn interaction rules such as *fiber AND e-check AND bill > $90*.
- $50 per offer and 12 months of value are assumptions; changing them in `params.yaml` moves the optimal campaign size.
- One snapshot of data: the model predicts *whether*, not *when*, a customer leaves.